# Обновление данных таблицы витрины с подсчетом системных флагов
## Постановка задачи
Имеется целевой DataFrame, построенный на основе таблицы t_entity в целевой базе данных Hive (db_target).  
На основе текщих данных источника сформирован новый DataFrame (идентичный по составу аттрибутов - t_entity (актуальный снимок данных на текущий день).

Требуется реализовать формирование результирующего DataFrame с вычислением системных колонок sys_oper_ts и sys_oper_type по следующим правилам:

- Для строк, которые были изменены — установить флаг U (обновление) и текущее время операции;
- Для новых строк — установить флаг I (вставка) и текущее время операции;
- Для строк, которые подлежат удалению — установить флаг D (удаление) и текущее время операции;
- Для строк, не изменившихся между stage и target — сохранить текущие значения системных колонок из целевого DataFrame.

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import $ivy.`org.apache.spark::spark-mllib:4.1.1` // если нужен ML
import $ivy.`org.apache.spark::spark-hive:4.1.1`

import $ivy.$
import $ivy.$
import $ivy.$

In [2]:
import scala.io.Source._
import scala.sys.process._

import scala.io.Source._
import scala.sys.process._

In [3]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.log4j.{Level, Logger}
// Подавить сообщения типа INFO при загрзке библиотек Scala
Logger.getLogger("org").setLevel(Level.WARN)
Logger.getLogger("akka").setLevel(Level.WARN)
// Spark - сессия
val spark = SparkSession.builder()
  .appName("Spark-notebook")
  .master("local[3]")
  .config("spark.driver.memory", "8g")
  .config("spark.sql.warehouse.dir", "spark-warehouse")
  .config("spark.sql.hive.metastore.version", "2.3.10") // Укажите версию вашего метастора Hive
  .config("hive.metastore.schema.verification", "2.3.10")
  .config("spark.ui.port", "4040")
  .enableHiveSupport()          
  .getOrCreate()

import spark.implicits._ // активация методов Saprk для объектов Scala

val sc = spark.sparkContext // Spatk - context
spark.sparkContext.setLogLevel("WARN") // Уровень сообщений - WARNING

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/09 21:10:59 WARN Utils: Your hostname, chul-PC, resolves to a loopback address: 127.0.1.1; using 192.168.88.252 instead (on interface eno1)
26/05/09 21:10:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/05/09 21:10:59 INFO SparkContext: Running Spark version 4.1.1
26/05/09 21:10:59 INFO SparkContext: OS info Linux, 7.0.0-15-generic, amd64
26/05/09 21:10:59 INFO SparkContext: Java version 17.0.18+8-Ubuntu-1
26/05/09 21:10:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/09 21:10:59 INFO ResourceUtils: ==============================================================
26/05/09 21:10:59 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/09 21:10:59 INFO ResourceUtils: ==============================================================
26/05/09 21:10:59 INFO SparkContext: Submit

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import org.apache.log4j.{Level, Logger}
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@728e51c2
import spark.implicits._
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@542ee26f

In [4]:
spark.version

res4: String = "4.1.1"

In [5]:
spark.sparkContext.uiWebUrl.get

res5: String = "http://192.168.88.252:4040"

In [6]:
import org.apache.spark.sql.expressions.Window
import scala.reflect.runtime.{universe => ru}
import org.apache.spark.sql.Row
import scala.collection.mutable.{WrappedArray,ArrayBuffer}
import java.io._
import scala.util.matching.Regex
import scala.sys.process._
import scala.xml._
import scala.math._
import org.apache.spark.sql.types.{StructType,StructField,StringType,IntegerType,DoubleType}
import java.time.LocalDateTime
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import spark.implicits._
import org.apache.spark.sql.functions.lead

import org.apache.spark.sql.expressions.Window
import scala.reflect.runtime.{universe => ru}
import org.apache.spark.sql.Row
import scala.collection.mutable.{WrappedArray,ArrayBuffer}
import java.io._
import scala.util.matching.Regex
import scala.sys.process._
import scala.xml._
import scala.math._
import org.apache.spark.sql.types.{StructType,StructField,StringType,IntegerType,DoubleType}
import java.time.LocalDateTime
import org.apache.spark.sql.functions._
import org.apache.spark.sql.DataFrame
import spark.implicits._
import org.apache.spark.sql.functions.lead

## Подготовка источников для теста и демонстрации  
 - создание тестовых БД ("сырой" источник, stage БД, целевой приемник)
 - создание тестовых тестовых таблиц для демонстрации

### DataBases

In [7]:
val db_source = "db_source"     // БД сырого источника
val db_stage = "db_stage"       // БД подготовленной таблицы (stage)
val db_target ="db_target" // БД целевой таблицы

db_source: String = "db_source"
db_stage: String = "db_stage"
db_target: String = "db_target"

In [8]:
// БД сырого источника
spark.sql(s"drop database if exists $db_source cascade")
spark.sql(s"create database $db_source")
// БД целевой таблицы
spark.sql(s"drop database if exists $db_target cascade")
spark.sql(s"create database $db_target")

spark.sql(s"drop database if exists $db_stage cascade")
spark.sql(s"create database $db_stage")

spark.sql("show databases").show(truncate = false)

26/05/09 21:11:05 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/05/09 21:11:05 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore chul@127.0.1.1
26/05/09 21:11:06 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/05/09 21:11:06 WARN ObjectStore: Failed to get database db_source, returning NoSuchObjectException
26/05/09 21:11:06 WARN ObjectStore: Failed to get database db_source, returning NoSuchObjectException
26/05/09 21:11:06 WARN ObjectStore: Failed to get database db_source, returning NoSuchObjectException
26/05/09 21:11:06 WARN TxnHandler: Cannot perform cleanup since metastore table does not exist
26/05/09 21:11:06 WARN ObjectStore: Failed to get database db_target, returning NoSuchObjectException
26/05/09 21:11:06 WARN ObjectStore: Failed to get database db_target, r

+---------+
|namespace|
+---------+
|db_source|
|db_stage |
|db_target|
|default  |
+---------+



res8_0: DataFrame = []
res8_1: DataFrame = []
res8_2: DataFrame = []
res8_3: DataFrame = []
res8_4: DataFrame = []
res8_5: DataFrame = []

### Источник из файла parquet
Колонка **for_test_type** содержит знчения для проведения возможности демонстрации функционала:  
 - for_update
 - for_insert
 - for_delete
 - nothing

In [9]:
val source_df = spark.read.parquet("data/source_for_test.parquet")
source_df.printSchema()

root
 |-- p_key_id: string (nullable = true)
 |-- int_field: integer (nullable = true)
 |-- cat_field: string (nullable = true)
 |-- status: string (nullable = true)
 |-- status_dt: string (nullable = true)
 |-- for_test_type: string (nullable = true)
 |-- relevance_flg: string (nullable = true)



source_df: DataFrame = [p_key_id: string, int_field: int ... 5 more fields]

In [10]:
source_df.groupBy("for_test_type").count().show()

+-------------+-----+
|for_test_type|count|
+-------------+-----+
|   for_update|   21|
|      nothing|   53|
|   for_insert|    3|
|   for_delete|    3|
+-------------+-----+



## Подготовка тествого примера
1) в БД target создается таблица **t_entity** с sys_oper_ts, sys_oper_type = "I" и for_test_type = "nothing". В эту таблицу загружаются строки с фильтром "for_test_type != 'for_insert'"
2) в БД target создается таблица **t_entity** В эту таблицу загружаются строки с фильтром "for_test_type != 'for_delete'"

In [11]:
source_df.where("for_test_type != 'for_insert'")
    .select("p_key_id","int_field","cat_field","status","status_dt","for_test_type","relevance_flg")
        .withColumn("sys_oper_ts",lit("2021-11-01").cast("timestamp"))
        .withColumn("sys_oper_type",lit("I").cast("string"))
        .withColumn("for_test_type",lit("nothing"))
        .write.format("parquet").mode("overwrite").saveAsTable(s"$db_target.t_entity")

spark.table(s"$db_target.t_entity").show(5,40)
println(s"count: ${spark.table(db_target + ".t_entity").count()}")

26/05/09 21:11:10 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/05/09 21:11:10 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


+--------+---------+---------+---------+----------+-------------+-------------+-------------------+-------------+
|p_key_id|int_field|cat_field|   status| status_dt|for_test_type|relevance_flg|        sys_oper_ts|sys_oper_type|
+--------+---------+---------+---------+----------+-------------+-------------+-------------------+-------------+
|   04324|  9418800|Обращение|Утвержден|22.05.2024|      nothing|        false|2021-11-01 00:00:00|            I|
|   04325|  9225479|Обращение|Утвержден|21.05.2024|      nothing|         true|2021-11-01 00:00:00|            I|
|   04326|  9349574|Обращение|Утвержден|02.05.2024|      nothing|         true|2021-11-01 00:00:00|            I|
|   04327|  9349574|Обращение|Утвержден|02.05.2024|      nothing|         true|2021-11-01 00:00:00|            I|
|   04328|  9441284|Обращение|Утвержден|21.05.2024|      nothing|         true|2021-11-01 00:00:00|            I|
+--------+---------+---------+---------+----------+-------------+-------------+---------

Таблица в БД stage с текущим snapshot

In [12]:
source_df.where("for_test_type != 'for_delete'")
.write.format("parquet").mode("overwrite").saveAsTable(s"$db_stage.t_entity")

spark.table(s"$db_stage.t_entity").show(5,40)
println(s"count: ${spark.table(db_stage + ".t_entity").count()}")

+--------+---------+---------+---------+----------+-------------+-------------+
|p_key_id|int_field|cat_field|   status| status_dt|for_test_type|relevance_flg|
+--------+---------+---------+---------+----------+-------------+-------------+
|   04325|  9225479|Обращение|Утвержден|21.05.2024|      nothing|         true|
|   04326|  9349574|Обращение|Утвержден|02.05.2024|      nothing|         true|
|   04327|  9349574|Обращение|Утвержден|02.05.2024|      nothing|         true|
|   04328|  9441284|Обращение|Утвержден|21.05.2024|   for_update|         true|
|   04329|  9441284|Обращение|Утвержден|21.05.2024|   for_update|         true|
+--------+---------+---------+---------+----------+-------------+-------------+
only showing top 5 rows
count: 77


## Функция расчета 
```scala
addSysOperValues (stg: DataFrame, pa: DataFrame, keys: Array[String], exclude_delete: Boolean = false ): DataFrame 
```

In [13]:
 /**
   * Сравинвает DataFrame stg отностильно DatFrame pa и формирует значения колонок 
       - sys_oper_ts - дата изменения/добавлени/записи записи
       - sys_oper_type - Тип операции: I - insert, U -update, D -delete
     В случае  exclude_delete = false - формирует в DataFrame pa строки с sys_oper_type = D, которые были удалены
   * @stg DataFrame - источник
   * @pa  DataFrame, относительно которого будут расчитаны флаги I,U,D
   * @keys Array с строковым названием колоок - клчей (по кторым будут опредеяться изменения)
   * @exclude_delete true - исключение операции удаление строк
 */
def addSysOperValues (stg: DataFrame, pa: DataFrame, 
                      keys: Array[String], 
                      exclude_delete: Boolean = false ): DataFrame = {
    val currentDate = current_timestamp() // системная дата
    // Результирубщий DataFrame
    val result = 
        if (stg.head(1).isEmpty) stg
        else {     // Если stage dataframe не пуст
            // UPDATE
            // для выражения when (колонки-ключи - не участвуют )
            // Колонки, кроме ключевых
            val nonKeyCols = stg.columns.filterNot(keys.contains)
            val allColsEqual = nonKeyCols.map(c => col(s"stg.$c") <=> col(s"pa.$c")).reduce(_ && _)
            // Условия, что все ключи pa или stg равны null
            val paNullCondition = keys.map(k => col(s"pa.$k").isNull).reduce(_ && _)
            val stgNullCondition = keys.map(k => col(s"stg.$k").isNull).reduce(_ && _)
            
            // Обновления и вставки
            val joined = stg.alias("stg").join(pa.alias("pa"), keys, "left")
            val upd = joined
                .withColumn("sys_oper_ts",
                     when(paNullCondition, currentDate)             // Новая запись из stg - INSERT
                    .when(!allColsEqual, currentDate)               // Есть изменения - UPDATE
                    .otherwise($"pa.sys_oper_ts")                   // Нет изменений
               ).withColumn("sys_oper_type",
                     when(paNullCondition, lit("I"))                // INSERT
                    .when(!allColsEqual, lit("U"))                  // UPDATE
                    .otherwise($"pa.sys_oper_type")                 // Без изменений
               ).select($"stg.*", $"sys_oper_ts", $"sys_oper_type")
            
          // DELETE - если нужно учитывать
            if (exclude_delete) upd
            else {
                val del = pa.join(stg, keys, "left_anti")
                      .withColumn("sys_oper_ts", 
                            when ($"sys_oper_type" === lit("D"), $"sys_oper_ts" )
                           .otherwise( currentDate))
                      .withColumn("sys_oper_type", lit("D"))
                upd.unionByName(del)
            }
        }
    
    result
}

defined function addSysOperValues

## Тест - пример

Колонка - ключ

In [14]:
val KEY = Array("p_key_id")

KEY: Array[String] = Array("p_key_id")

In [15]:
val stg = spark.table(s"$db_stage.t_entity") // stage - DataFrame
val pa = spark.table(s"$db_target.t_entity") // Текущий target - DataFrame

println(s"stage: ${stg.count()}")
println(s"target: ${pa.count()}")

stage: 77
target: 77


stg: DataFrame = [p_key_id: string, int_field: int ... 5 more fields]
pa: DataFrame = [p_key_id: string, int_field: int ... 7 more fields]

Применение функции (с учетом операции удаления)

In [16]:
val df_result = addSysOperValues(stg = stg, pa= pa ,keys = KEY)
println(s"result: ${df_result.count()}")

result: 80


df_result: DataFrame = [p_key_id: string, int_field: int ... 7 more fields]

In [17]:
df_result.show(10,false)

+--------+---------+---------+---------+----------+-------------+-------------+--------------------------+-------------+
|p_key_id|int_field|cat_field|status   |status_dt |for_test_type|relevance_flg|sys_oper_ts               |sys_oper_type|
+--------+---------+---------+---------+----------+-------------+-------------+--------------------------+-------------+
|04325   |9225479  |Обращение|Утвержден|21.05.2024|nothing      |true         |2021-11-01 00:00:00       |I            |
|04326   |9349574  |Обращение|Утвержден|02.05.2024|nothing      |true         |2021-11-01 00:00:00       |I            |
|04327   |9349574  |Обращение|Утвержден|02.05.2024|nothing      |true         |2021-11-01 00:00:00       |I            |
|04328   |9441284  |Обращение|Утвержден|21.05.2024|for_update   |true         |2026-05-09 21:11:12.911876|U            |
|04329   |9441284  |Обращение|Утвержден|21.05.2024|for_update   |true         |2026-05-09 21:11:12.911876|U            |
|04331   |9394803  |Обращение|Ут

In [18]:
df_result.groupBy("sys_oper_type","sys_oper_ts").count().show(false)

+-------------+--------------------------+-----+
|sys_oper_type|sys_oper_ts               |count|
+-------------+--------------------------+-----+
|U            |2026-05-09 21:11:13.331975|21   |
|I            |2026-05-09 21:11:13.331975|3    |
|I            |2021-11-01 00:00:00       |53   |
|D            |2026-05-09 21:11:13.331975|3    |
+-------------+--------------------------+-----+



Применение функции (без учета операции удаления)

In [19]:
val df_result_wo_D = addSysOperValues(stg = stg, pa= pa, keys = KEY, exclude_delete = true)
println(s"result: ${df_result_wo_D.count()}")

result: 77


df_result_wo_D: DataFrame = [p_key_id: string, int_field: int ... 7 more fields]

In [20]:
df_result_wo_D.show(10,false)

+--------+---------+---------+---------+----------+-------------+-------------+--------------------------+-------------+
|p_key_id|int_field|cat_field|status   |status_dt |for_test_type|relevance_flg|sys_oper_ts               |sys_oper_type|
+--------+---------+---------+---------+----------+-------------+-------------+--------------------------+-------------+
|04325   |9225479  |Обращение|Утвержден|21.05.2024|nothing      |true         |2021-11-01 00:00:00       |I            |
|04326   |9349574  |Обращение|Утвержден|02.05.2024|nothing      |true         |2021-11-01 00:00:00       |I            |
|04327   |9349574  |Обращение|Утвержден|02.05.2024|nothing      |true         |2021-11-01 00:00:00       |I            |
|04328   |9441284  |Обращение|Утвержден|21.05.2024|for_update   |true         |2026-05-09 21:11:25.330949|U            |
|04329   |9441284  |Обращение|Утвержден|21.05.2024|for_update   |true         |2026-05-09 21:11:25.330949|U            |
|04331   |9394803  |Обращение|Ут

In [21]:
df_result_wo_D.groupBy("sys_oper_type","sys_oper_ts").count().show(false)

+-------------+--------------------------+-----+
|sys_oper_type|sys_oper_ts               |count|
+-------------+--------------------------+-----+
|I            |2026-05-09 21:11:31.831807|3    |
|U            |2026-05-09 21:11:31.831807|21   |
|I            |2021-11-01 00:00:00       |53   |
+-------------+--------------------------+-----+



In [22]:
spark.stop()